# QLoRA Fine-tuning: Mistral-7B on Research Paper QA
Fine-tunes Mistral-7B-Instruct-v0.2 using 4-bit quantization + LoRA.
**Requires: Colab Pro (A100) or Kaggle (T4/P100)**

Expected training time: ~45 min on A100, ~2.5 hrs on T4

In [ ]:
!pip install -q peft trl bitsandbytes accelerate transformers datasets

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import sys
sys.path.insert(0, '..')
from src.finetuning.dataset_prep import prepare_dataset, SEED_QA_PAIRS

# Use seed data or load your own QA pairs
# For real training: collect 500-1000 QA pairs from your paper corpus
paths = prepare_dataset(SEED_QA_PAIRS, output_path='data/finetune_dataset.jsonl')
print('Dataset ready:', paths)

In [ ]:
from src.finetuning.train_qlora import train, QLoRAConfig

cfg = QLoRAConfig(
    lora_r=16,
    lora_alpha=32,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
)

train(
    train_file=paths['train'],
    val_file=paths['val'],
    output_dir='models/mistral-qlora-adapter',
    config=cfg,
)

In [ ]:
# ── Test fine-tuned adapter in RAG pipeline ───────────────────────────────────
from src.pipeline.rag_chain import RAGPipeline

rag_finetuned = RAGPipeline(
    index_path='indexes/faiss',
    adapter_path='models/mistral-qlora-adapter',  # load our adapter
)
rag_finetuned.build_index(pdf_dir='data/papers', save_index=False)

result = rag_finetuned.query('What methodology does this paper propose?')
print('Answer:', result['result'])